# Deep Learning Midterm Notebook: Qwen 2B LoRA for Text-to-SVG (Kaggle)

Author: Thomas Kong

NetId: tk2558

Goal: provide a practical scaffold for Qwen-2B-class fine-tuning + submission generation.

## Referenced Data and Docs

### Dataset resources
- Provided train.csv

### Qwen 2B fine-tuning references
- Unsloth Qwen fine-tune docs: https://unsloth.ai/docs/models/qwen3.5/fine-tune
- Qwen3.5-2B Vision notebook: https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(2B)_Vision.ipynb


### **Section 0: Installing Necessary Packages**

> Make sure all packages and their versions are available and correct

> Uncomment whatever is needed to install for notebook environment

> Make sure train.csv and test.csv is uploaded for notebook to access



In [ ]:
# # Uncomment the following in a fresh Kaggle notebook environment.
#%pip install -q unsloth datasets trl transformers==4.56.2 accelerate peft bitsandbytes pandas lxml ftfy svgpathtools

# # Install Node.js (if not already available)
# !apt-get update -y
# !apt-get install -y nodejs npm

# # Install SVGO
#!npm install -g svgo

In [ ]:
import unsloth, transformers, trl
# CHECK AVAILABLE AND CORRECT VERSIONS
print(transformers.__version__)
print(trl.__version__)
print(unsloth.__version__)

In [ ]:
# Install Node.js (if not already available)
!node -v # Verify Node
!svgo --version # Verify SVGO installation

### **Section 1: Configuration**

> Initialize variables for the notebook and models

> After running all cellblocks in Section 1, you can skip to 2B if you are already using pre-installed training_compressed.csv or skip to Section 7 if you are using pretrained fine-tuned model provided.


In [ ]:
import os
import re
import time
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

from datasets import concatenate_datasets, load_dataset, Dataset
import hashlib, random, numpy as np, torch

NETID = "tk2558"
SEED  = int(hashlib.sha256(NETID.encode()).hexdigest(), 16) % 10000
print(f"NetID: {NETID}  |  Seed: {SEED}")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Core training config.
CONFIG = {
    "model_name": "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit",  # Verify exact ID from the linked Unsloth notebook.
    "max_seq_length": 2048,
    "lora_r": 16,
    "lora_alpha": 64, # 32, #16,
    "learning_rate": 2e-4,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 16, #4,
    "warmup_ratio": 0.05,
    "warmup_steps": 100,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 200,
    "max_train_samples_per_source": 40000,
    "eval_size": 0.05, #0.02,
    #"output_dir": "/kaggle/working/qwen2b_svg_lora",
    "output_dir": "/content/working/qwen2b_svg_lora",
}

CONFIG

In [ ]:
import pandas as pd

# LOAD DATASET
df = pd.read_csv("/content/train.csv", engine="python", on_bad_lines='skip')
assert all(col in df.columns for col in ["id", "prompt", "svg"])

print("Total samples:", len(df))
df = df.dropna(subset=["prompt", "svg"])
df.head()

### **Section 2A: Preprocessing Data Part 1**

> (Can skip this part if you are using pre-installed train_compression.csv)

> This part is necessary for creating train_compression.csv, a more optimized and streamline version of the train.csv for model to train. This is the Data Compression Pipeline

> In this Data Compression Pipeline we: compress SVG to use less tokens and scale every SVG to fill a 256x256 Canvas


In [ ]:
with open("/content/svgo.config.transform.js", "w") as f:
    f.write("""
export default {
  multipass: true,
  floatPrecision: 4,
  plugins: [
    {
      name: "preset-default",
      params: {
        overrides: {
          removeViewBox: false,
          removeDimensions: false,
          mergePaths: false,
          convertShapeToPath: false,
          cleanupNumericValues: false
        }
      }
    },
    {
      name: "convertTransform"
    },
    {
      name: "sortAttrs"
    }
  ]
};
""")

print("svgo.config.transform.js FILE CREATED")

In [ ]:
with open("/content/svgo.config.compress.js", "w") as f:
    f.write("""
export default {
  multipass: true,
  floatPrecision: 2,
  plugins: [
    {
      name: "preset-default",
      params: {
        overrides: {
          removeViewBox: false,
          removeDimensions: false,
          mergePaths: false,
          convertShapeToPath: false
        }
      }
    },

    {
      name: "cleanupNumericValues",
      params: { floatPrecision: 2 }
    },

    {
      name: "sortAttrs"
    }
  ]
};
""")

print("svgo.config.compress.JS FILE CREATED")

In [ ]:
import os

BASE_DIR = "/content/svg_pipeline"

IN_DIR = f"{BASE_DIR}/svg_in"
MID_DIR = f"{BASE_DIR}/svg_mid"
OUT_DIR = f"{BASE_DIR}/svg_out"
FINAL_DIR = f"{BASE_DIR}/svg_final"

os.makedirs(IN_DIR, exist_ok=True)
os.makedirs(MID_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FINAL_DIR, exist_ok=True)

print("FOLDER DIRECTORY CREATED")

In [ ]:
import re
import subprocess
import tempfile
import os
from transformers import AutoTokenizer

# SVG cleanup
def basic_svg_cleanup(svg):
    if not isinstance(svg, str):
        return ""

    svg = re.sub(r"<\?xml.*?\?>", "", svg) # Remove XML header if exists
    svg = re.sub(r"<!--.*?-->", "", svg, flags=re.DOTALL) # Remove comments
    svg = re.sub(r"<metadata.*?</metadata>", "", svg, flags=re.DOTALL) # Remove metadata blocks
    svg = re.sub(r"\s+", " ", svg) # Normalize whitespace
    svg = re.sub(r">\s+<", "><", svg) # Remove spaces between tags
    svg = re.sub(r"\s*=\s*", "=", svg) # Remove unnecessary quotes spacing

    return svg.strip()


tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
def count_svg_tokens(svg):
    return len(tokenizer(svg)["input_ids"])

In [ ]:
import re
from svgpathtools import parse_path, Path
from lxml import etree

def _parse_translate(transform_str):
    # Extract (tx, ty) from a translate(...) transform string. Returns (0,0) if not found
    m = re.search(r'translate\(([^)]+)\)', transform_str or '')
    if not m:
        return 0.0, 0.0
    nums = re.findall(r'-?[\d.]+', m.group(1))
    tx = float(nums[0]) if len(nums) > 0 else 0.0
    ty = float(nums[1]) if len(nums) > 1 else 0.0
    return tx, ty


def bake_path_translate(svg_string):
    try:
        root = etree.fromstring(svg_string.encode())
    except etree.XMLSyntaxError:
        return _bake_path_translate_regex(svg_string)

    def _walk(el, acc_tx, acc_ty):
        tag = etree.QName(el.tag).localname if '{' in el.tag else el.tag
        transform = el.get('transform', '')

        # Accumulate translate from this element's own transform
        own_tx, own_ty = _parse_translate(transform) if 'translate' in transform else (0.0, 0.0)
        total_tx = acc_tx + own_tx
        total_ty = acc_ty + own_ty

        if tag == 'path':
            d = el.get('d', '')
            if d and (total_tx != 0.0 or total_ty != 0.0):
                try:
                    baked = parse_path(d).translated(complex(total_tx, total_ty))
                    el.set('d', baked.d())
                except Exception:
                    pass

            # Remove the translate part from the path's OWN transform (if any).
            if own_tx != 0.0 or own_ty != 0.0:
                remaining = re.sub(r'\s*translate\([^)]+\)', '', transform).strip()
                if remaining:
                    el.set('transform', remaining)
                else:
                    del el.attrib['transform']

            # Paths are leaves for translation purposes; reset accumulator for any children
            for child in el:
                _walk(child, 0.0, 0.0)

        elif tag == 'g':
            # Process children first so they absorb the full accumulated translation
            for child in el:
                _walk(child, total_tx, total_ty)
            if own_tx != 0.0 or own_ty != 0.0:
                remaining = re.sub(r'\s*translate\([^)]+\)', '', transform).strip()
                if remaining:
                    el.set('transform', remaining)
                elif 'transform' in el.attrib:
                    del el.attrib['transform']

        else:
            # For all other elements: accumulate but do not remove the transform.
            for child in el:
                _walk(child, total_tx, total_ty)

    _walk(root, 0.0, 0.0)
    return etree.tostring(root, encoding='unicode')


def _bake_path_translate_regex(svg_string):
    # Fallback for malformed SVGs lxml cannot parse — handles only standalone <path transform=translate>
    def bake(m):
        full_tag = m.group(0)
        t_match = re.search(r'transform="translate\(([^)]+)\)"', full_tag)
        d_match = re.search(r'(?<![a-z])d="([^"]+)"', full_tag)
        if not t_match or not d_match:
            return full_tag
        nums = re.findall(r'-?[\d.]+', t_match.group(1))
        tx = float(nums[0]) if len(nums) > 0 else 0.0
        ty = float(nums[1]) if len(nums) > 1 else 0.0
        try:
            path = parse_path(d_match.group(1))
            baked = path.translated(complex(tx, ty))
            new_tag = re.sub(r'(?<![a-z])d="[^"]+"', f'd="{baked.d()}"', full_tag)
            new_tag = re.sub(r'\s*transform="translate\([^)]+\)"', '', new_tag)
            return new_tag
        except Exception:
            return full_tag
    return re.sub(r'<path\b[^>]*(?:/>|>)', bake, svg_string)


def normalize_paths(svg_string):
    if re.search(r'<g[^>]+transform=[^>]*translate', svg_string):
        return svg_string

    paths = re.findall(r'(?<![a-z])d="([^"]+)"', svg_string)
    if not paths:
        return svg_string

    xmin, ymin = float('inf'), float('inf')
    parsed_paths = []
    valid_old_ds = []

    for d in paths:
        try:
            p = parse_path(d)
        except Exception:
            continue
        bbox = p.bbox()
        if bbox[0] == bbox[1] and bbox[2] == bbox[3]:
            svg_string = re.sub(
                r'(?<![a-z])d="' + re.escape(d) + r'"',
                'd=""', svg_string, count=1,
            )
            continue
        xmin = min(xmin, bbox[0])
        ymin = min(ymin, bbox[2])
        parsed_paths.append(p)
        valid_old_ds.append(d)

    if xmin == float('inf'):
        return svg_string

    # Only shift if geometry actually starts off-canvas
    if xmin >= 0 and ymin >= 0:
        return svg_string

    for old_d, p in zip(valid_old_ds, parsed_paths):
        new_p = p.translated(complex(-xmin, -ymin))
        svg_string = re.sub(
            r'(?<![a-z])d="' + re.escape(old_d) + r'"',
            f'd="{new_p.d()}"', svg_string, count=1,
        )

    return svg_string

In [ ]:
import re
from svgpathtools import parse_path, Path

def scale_svg_to_256(svg_string):
    """
    Scale an SVG so its largest dimension fills 256 px, updating all geometry.
    Call normalize_paths() BEFORE this function, not inside it.
    """

    # -- 1. Parse canvas size -------------------------------------------- #
    viewbox_match = re.search(
        r'viewBox="(-?[\d.]+)\s+(-?[\d.]+)\s+([\d.]+)\s+([\d.]+)"',
        svg_string,
    )
    if viewbox_match:
        orig_w = float(viewbox_match.group(3))
        orig_h = float(viewbox_match.group(4))
    else:
        w_match = re.search(r'width="([\d.]+)"', svg_string)
        h_match = re.search(r'height="([\d.]+)"', svg_string)
        orig_w = float(w_match.group(1)) if w_match else 256.0
        orig_h = float(h_match.group(1)) if h_match else 256.0

    scale = 256.0 / max(orig_w, orig_h)
    new_w  = round(orig_w * scale, 4)
    new_h  = round(orig_h * scale, 4)

    def fmt(v):
        return int(v) if v == int(v) else round(v, 4)

    # -- 2. Scale transform coordinates ---------------------------------- #
    def scale_translate(m):
        nums = re.findall(r"-?[\d.]+", m.group(1))
        tx = round(float(nums[0]) * scale, 4) if nums else 0
        ty = round(float(nums[1]) * scale, 4) if len(nums) > 1 else 0
        return f"translate({tx} {ty})"

    def scale_rotate(m):
        nums = re.findall(r"-?[\d.]+", m.group(1))
        angle = nums[0]                          # angle: never scaled
        if len(nums) == 3:
            cx = round(float(nums[1]) * scale, 4)
            cy = round(float(nums[2]) * scale, 4)
            return f"rotate({angle} {cx} {cy})"
        return f"rotate({angle})"

    def scale_matrix(m):
        nums = re.findall(r"-?[\d.]+", m.group(1))
        if len(nums) == 6:
            a, b, c, d, e, f = nums
            e2 = round(float(e) * scale, 4)      # e: x-translation
            f2 = round(float(f) * scale, 4)      # f: y-translation
            return f"matrix({a} {b} {c} {d} {e2} {f2})"
        return m.group(0)

    svg_string = re.sub(r"translate\(([^)]+)\)", scale_translate, svg_string)
    svg_string = re.sub(r"rotate\(([^)]+)\)",    scale_rotate,    svg_string)
    svg_string = re.sub(r"matrix\(([^)]+)\)",    scale_matrix,    svg_string)

    # -- 3. Scale path d= (command-aware, arc flags preserved) ----------- #
    # (?<![a-z]) prevents matching id=, method=, href= as path data
    def scale_path_d(m):
        return f'd="{_scale_path_commands(m.group(1), scale)}"'
    svg_string = re.sub(r'(?<![a-z])d="([^"]+)"', scale_path_d, svg_string)

    # -- 4. Scale polyline/polygon points= ------------------------------- #
    def scale_points(m):
        def sn(n): return str(round(float(n.group()) * scale, 4))
        return f'points="{re.sub(r"-?[\d.]+", sn, m.group(1))}"'
    svg_string = re.sub(r'points="([^"]+)"', scale_points, svg_string)

    # -- 5. Scale presentation attributes -------------------------------- #
    def make_scaler(attr):
        def r(m): return f'{m.group(1)}="{fmt(float(m.group(2)) * scale)}"'
        return r

    # Compound attrs first — prevents (x)= from double-scaling cx/dx/rx etc.
    for attr in ["cx", "cy", "rx", "ry", "dx", "dy", "x1", "y1", "x2", "y2"]:
        svg_string = re.sub(fr'({attr})="(-?[\d.]+)"', make_scaler(attr), svg_string)

    svg_string = re.sub(r'(?<![a-z])(x)="(-?[\d.]+)"',          make_scaler("x"), svg_string)
    svg_string = re.sub(r'(?<![a-z])(y)="(-?[\d.]+)"',          make_scaler("y"), svg_string)
    svg_string = re.sub(r'(?<![a-z])(r)(?![a-z])="(-?[\d.]+)"', make_scaler("r"), svg_string)

    for attr in ["width", "height"]:  # stroke-width caught here too
        svg_string = re.sub(fr'({attr})="(-?[\d.]+)"', make_scaler(attr), svg_string)

    svg_string = re.sub(r'(font-size)="(-?[\d.]+)"', make_scaler("font-size"), svg_string)

    # -- 6. Always update viewBox ---------------------------------------- #
    new_vb = f"0 0 {fmt(new_w)} {fmt(new_h)}"
    if viewbox_match:
        svg_string = re.sub(
            r'(<svg[^>]*\s)viewBox="[^"]*"',
            lambda m: f'{m.group(1)}viewBox="{new_vb}"',
            svg_string, count=1,
        )

    # -- 7. Force root width/height -------------------------------------- #
    svg_string = re.sub(
        r'(<svg[^>]*\s)width="[^"]*"',
        lambda m: f'{m.group(1)}width="{fmt(new_w)}"',
        svg_string, count=1,
    )
    svg_string = re.sub(
        r'(<svg[^>]*\s)height="[^"]*"',
        lambda m: f'{m.group(1)}height="{fmt(new_h)}"',
        svg_string, count=1,
    )

    # Clean up empty paths left by normalize_paths
    svg_string = re.sub(r'<path[^>]*d=""[^>]*/>', '', svg_string)
    svg_string = re.sub(r'<path[^>]*d="\s*"[^>]*></path>', '', svg_string)

    return svg_string


def _scale_path_commands(d: str, scale: float) -> str:
    # Command-aware path scaler. Arc flags are never multiplied by scale.
    tokens = re.findall(
        r"[MmZzLlHhVvCcSsQqTtAa]"
        r"|[-+]?(?:\d+\.?\d*|\.\d+)(?:[eE][-+]?\d+)?",
        d,
    )
    out = []
    i   = 0

    def s(v): return str(round(float(v) * scale, 4))

    while i < len(tokens):
        tok = tokens[i]
        if tok in "Zz":
            out.append(tok); i += 1
        elif tok in "MmLlTt":
            out.append(tok); i += 1
            out.append(s(tokens[i])); i += 1
            out.append(s(tokens[i])); i += 1
        elif tok in "Hh":
            out.append(tok); i += 1
            out.append(s(tokens[i])); i += 1
        elif tok in "Vv":
            out.append(tok); i += 1
            out.append(s(tokens[i])); i += 1
        elif tok in "Cc":
            out.append(tok); i += 1
            for _ in range(6): out.append(s(tokens[i])); i += 1
        elif tok in "SsQq":
            out.append(tok); i += 1
            for _ in range(4): out.append(s(tokens[i])); i += 1
        elif tok in "Aa":
            out.append(tok); i += 1
            out.append(s(tokens[i])); i += 1   # rx       -- scale
            out.append(s(tokens[i])); i += 1   # ry       -- scale
            out.append(tokens[i]);    i += 1   # rotation -- NO scale
            out.append(tokens[i]);    i += 1   # large-arc -- NO scale
            out.append(tokens[i]);    i += 1   # sweep    -- NO scale
            out.append(s(tokens[i])); i += 1   # x        -- scale
            out.append(s(tokens[i])); i += 1   # y        -- scale
        else:
            out.append(s(tok)); i += 1         # implicit repeat

    return " ".join(out)

In [ ]:
from svgpathtools import parse_path, Path

def clean_degenerate_segments(svg_string):
    # Remove zero-length segments from all paths
    def clean_path(match):
        d = match.group(1)
        try:
            path = parse_path(d)
        except Exception:
            return match.group(0)
        new_segs = [seg for seg in path if abs(seg.start - seg.end) >= 1e-6]
        if not new_segs:
            return match.group(0)
        return f'd="{Path(*new_segs).d()}"'
    # (?<![a-z]) guard prevents matching id=, href= etc.
    return re.sub(r'(?<![a-z])d="([^"]+)"', clean_path, svg_string)


def round_path_numbers(svg_string, precision=2):
    """Round all floats in the SVG to reduce token count."""
    def round_num(m):
        return str(round(float(m.group()), precision))
    return re.sub(r'(?<![a-zA-Z])-?\d+\.\d+', round_num, svg_string)


def remove_redundant_transforms(svg_string):
    def remove(m):
        nums = re.findall(r'-?[\d.]+', m.group(1))
        if all(abs(float(n)) < 1e-6 for n in nums):
            return ''       # zero translate -> drop it
        return m.group(0)  # non-zero -> keep it

    return re.sub(r'\s*transform="(translate\([^)]+\))"', remove, svg_string)


In [ ]:
# COMPRESSION PIPELINE A (BASIC CLEAN UP)
from tqdm import tqdm

valid_indices = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    svg = basic_svg_cleanup(row["svg"]) # BASIC CLEANUP
    if not svg.startswith("<svg"):
        continue

    file_path = f"{IN_DIR}/{i}.svg"

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(svg)

    valid_indices.append(i)

print("Saved SVGs:", len(valid_indices))

In [ ]:
%%capture
# Just to ignore Streaming Output
!svgo -f /content/svg_pipeline/svg_in \
      -o /content/svg_pipeline/svg_mid \
      --multipass \
      --config=/content/svgo.config.transform.js

In [ ]:
# COMPRESSION PIPELINE B (SCALE)

scaled_indices = []
failed_indices = []

for i in valid_indices:
    path = f"{MID_DIR}/{i}.svg"
    if not os.path.exists(path):
        continue
    with open(path, "r", encoding="utf-8") as f:
        svg = f.read()

    try:
        svg = bake_path_translate(svg)              # 1. absorb any translate() SVGO missed
        svg = normalize_paths(svg)                  # 2. shift to (0,0) BEFORE scaling
        svg = scale_svg_to_256(svg)                 # 3. scale to 256 canvas
        svg = clean_degenerate_segments(svg)        # 4. remove zero-length segments
        svg = remove_redundant_transforms(svg)      # 5. drop zero-value transforms
        svg = round_path_numbers(svg, precision=2)  # 6. round for token savings

    except Exception as e:
        failed_indices.append((i, str(e)))
        continue

    out_path = f"{OUT_DIR}/{i}.svg"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(svg)

    scaled_indices.append(i)

print(f"Scaled:  {len(scaled_indices)}")
print(f"Failed:  {len(failed_indices)}")

In [ ]:
%%capture
# Just to ignore Streaming Output
!svgo -f /content/svg_pipeline/svg_out \
      -o /content/svg_pipeline/svg_final \
      --multipass \
      --config=/content/svgo.config.compress.js

In [ ]:
optimized_map = {}

for i in valid_indices:
    path = f"{FINAL_DIR}/{i}.svg"

    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            svg = f.read()

            # Basic validation
            if svg.startswith("<svg") and svg.endswith("</svg>"):
                optimized_map[i] = svg

In [ ]:
MAX_SVG_TOKENS = 1024 #512
compressed_rows = []

for i, row in df.iterrows():
    if i not in optimized_map:
        continue

    svg = optimized_map[i]
    token_len = count_svg_tokens(svg)

    if token_len <= MAX_SVG_TOKENS:
        compressed_rows.append({
            "id": row["id"],
            "prompt": row["prompt"],
            "svg": svg,
            "tokens": token_len
        })

In [ ]:
compressed_df = pd.DataFrame(compressed_rows)

print("Original:", len(df))
print("After compression:", len(compressed_df))
print("\nToken stats:")
print(compressed_df["tokens"].describe())

In [ ]:
compressed_df.to_csv("/content/train_compressed.csv", index=False)

### **Section 2B: Preprocessing Data Part 2**

> (Can skip to  part if you are using pre-installed train_compression.csv)

> Now that we have train_compression.csv, we need to clean the data some more. If you look deeper at train_compression.csv you can noticed there are still some bad data (incorrect SVG outputs, encoding garbage in prompts, etc). This is the Data Cleaning Pipeline which helps to provide model better training data

In [ ]:
# Data Catalog
# Can just skip to this part if using pre-installed train_compression.csv

DATASET_CATALOG = {
    "kaggle/train_compressed.csv": {
        "type": "csv",
        #"path": "/kaggle/input/datasets/tk2558/train-prompt/train_compressed.csv",
        "path": "/content/train_compressed.csv",
        "prompt_fields": ["prompt"],
        "svg_fields": ["svg"],
    },
}

ACTIVE_SOURCES = [
    "kaggle/train_compressed.csv", # ONLY ALLOWED TO USE KAGGLE DATASET
]

In [ ]:
from unsloth import FastLanguageModel
# Load Model (4bit for efficiency)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

print("\n=== CHECK TOKENS ===")
print("EOS token:", tokenizer.eos_token)       # should be <|im_end|>
print("EOS token ID:", tokenizer.eos_token_id) # should be 151645
print("PAD token:", tokenizer.pad_token)
print("Padding side:", tokenizer.padding_side)
print("==================\n")

# Apply LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

In [ ]:
def is_solid_color_prompt(prompt):
    p = prompt.lower()
    patterns = [
        "The image consists of a single solid",
        "The image contains a single solid",
        "entire visible area",
        "entire frame",
        "entire space",
        "entire area",
        "with no additional elements or features",
    ]

    return any(pattern in p for pattern in patterns)

# 2. EXTRACT COLOR FROM PROMPT
def extract_color(prompt):
    p = prompt.lower()
    colors = ["black", "white", "red", "blue", "green", "yellow", "orange", "purple", "gray", "grey"]
    for c in colors:
        if c in p:
            if c == "grey":
                return "gray"
            return c
    return "black" # fallback

# 3. GENERATE CORRECT SVG
def generate_solid_svg(color):
    return f'''<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 256 256"><rect fill="{color}" width="256" height="256"/></svg>'''


In [ ]:
def _pick_first_non_empty(example, keys):
    for key in keys:
        if key in example and example[key] is not None:
            val = str(example[key]).strip()
            if val:
                return val
    return ""

def clean_svg(svg, prompt): # SECOND ROUND OF CLEANUP JUST IN CASE
    svg = svg.strip()

    # Remove scripts / bad stuff
    svg = re.sub(r"<script.*?>.*?</script>", "", svg, flags=re.DOTALL)

    # Remove invalid tags (optional but helpful)
    svg = re.sub(r"</?(foreignObject|iframe|object).*?>", "", svg)

    # Ensure proper root
    if not svg.lower().startswith("<svg"):
        #print("SVG_start_missing")
        return ""

    # Ensure closing tag
    if not svg.endswith("</svg>"):
        #print("SVG_end_missing")
        return ""

    if len(svg) > 7999: # SVG LESS THAN 8000 CHARACTERS
        #print("SVG_too_big")
        return ""

    if len(prompt) > 800: # SUSPICIOUS PROMPT IF TOO LONG
        #print("SVG_too_big")
        return ""

    if svg.count("<path") > 255: # LESS THAN 256 PATHS
        #print("SVG_too_many_paths")
        return ""

    if is_solid_color_prompt(prompt):  # NOTICED THAT ALL SINGLE SOLID COLOR TRAIN PROMPTS ARE INACCURATE. FIX THEM!
        color = extract_color(prompt)
        new_svg = generate_solid_svg(color)
        return new_svg

    # remove <title>...</title> and <desc>...</desc> (GET RID OF EXTRA NOISE IN DATA)
    svg = re.sub(r"<desc.*?>.*?</desc>", "", svg, flags=re.DOTALL | re.IGNORECASE)
    svg = re.sub(r"<title.*?>.*?</title>", "", svg, flags=re.DOTALL | re.IGNORECASE)
    return svg

def clean_training_prompt(prompt: str) -> str:
    if not isinstance(prompt, str):
        return ""

    # 1. Fix encoding garbage (ÃƒÂ‚Ã‚Â etc.)
    try:
        import ftfy
        prompt = ftfy.fix_text(prompt)
    except ImportError: # fallback without ftfy — catches the most common double-encoding
        try:
            prompt = prompt.encode('latin-1').decode('utf-8')
        except (UnicodeDecodeError, UnicodeEncodeError):
            pass

    # 2. Remove meta-instructions that leaked into prompts
    meta_patterns = [
        r"don'?t use markdown[^.]*\.?",
        r"just give svg code[^.]*\.?",
    ]

    for pattern in meta_patterns:
        prompt = re.sub(pattern, '', prompt, flags=re.IGNORECASE)

    # 3. Remove leftover punctuation/whitespace from deletions
    prompt = re.sub(r'\s+', ' ', prompt).strip().strip('.,;:')

    return prompt


def should_keep_training_prompt(prompt: str) -> bool:
    if not prompt or len(prompt.strip()) < 5:
        return False

    garbage_chars = ['Ã', 'Â', 'â€', 'Ä', 'Å', '\x00'] # Reject if still has encoding artifacts after cleaning
    if any(c in prompt for c in garbage_chars):
        return False

    # Reject if the entire prompt is just a meta-instruction
    meta_only = re.match(
        r"^(don'?t use|just give|only return|output only|no markdown)",
        prompt.strip(), re.IGNORECASE
    )
    if meta_only:
        return False

    return True


def to_prompt_svg(example, prompt_fields, svg_fields):
    prompt = _pick_first_non_empty(example, prompt_fields)
    svg = _pick_first_non_empty(example, svg_fields)

    prompt = clean_training_prompt(prompt)
    if not should_keep_training_prompt(prompt):
        return {"prompt": "", "svg": ""}

    svg = clean_svg(svg, prompt)

    if not svg.lower().startswith("<svg"):
        return {"prompt": "", "svg": ""}

    return {"prompt": prompt, "svg": svg}

def load_source_dataset(dataset_id, cfg, max_samples):
    print(f"Loading {dataset_id} ...")

    # CASE 1: HuggingFace dataset (if using external Datasets)
    if cfg.get("type", "hf") == "hf":
        if "data_files" in cfg:
            ds = load_dataset(
                dataset_id,
                data_files=cfg["data_files"],
                split=cfg["split"]
            )
        else:
            ds = load_dataset(dataset_id, split=cfg["split"])

    # CASE 2: Kaggle CSV
    elif cfg["type"] == "csv":
        #df = pd.read_csv(cfg["path"], engine="python", on_bad_lines="skip")
        df = pd.read_csv(cfg["path"])
        ds = Dataset.from_pandas(df)

    else:
        raise ValueError(f"Unknown dataset type for {dataset_id}")

    if max_samples and len(ds) > max_samples:
        ds = ds.shuffle(seed=SEED).select(range(max_samples))
    ds = ds.map(
        lambda ex: to_prompt_svg(ex, cfg["prompt_fields"], cfg["svg_fields"]),
        remove_columns=ds.column_names,
        desc=f"normalizing {dataset_id}",
    )
    ds = ds.filter(lambda x: bool(x["prompt"]) and bool(x["svg"]))
    print(f"{dataset_id}: {len(ds)} usable rows")
    return ds

print("READY")

### **Section 3 (IMPORTANT): Secret Key for Hugging Face API**

> Make sure notebook can access the secret key/token and you can assign it to an environment variable.

> If using Kaggle, uncomment top half of codeblock and comment the bottom. If using Google Colab, uncomment bottom half of codeblock and comment the top.

In [ ]:
# from kaggle_secrets import UserSecretsClient
# import os
# user_secrets = UserSecretsClient()

# # Set the HF_TOKEN environment variable
# os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

# ----------------------------------------------------------------- #

from google.colab import userdata
import os

# Access the secret and assign it to an environment variable
# Replace 'MY_API_KEY' with the exact name you used in the Secrets Manager
api_key = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = api_key

print("READY")

### **Section 4: Load Dataset**

> Normalize Dataset and format it to SFT training style so it's ready for the model to learn from. Double check everything is functional!

In [ ]:
datasets_ok = []
datasets_origin = []

for source in ACTIVE_SOURCES:
    try:
        ds = load_source_dataset(
            source,
            DATASET_CATALOG[source],
            CONFIG["max_train_samples_per_source"],
        )
        datasets_ok.append(ds)
        datasets_origin.append((source, ds))
    except Exception as e:
        print(f"Skipping {source}: {type(e).__name__}: {e}")

if not datasets_ok:
    raise RuntimeError("No dataset loaded. Check dataset IDs, internet access, and schema fields.")

train_raw = datasets_ok[0] if len(datasets_ok) == 1 else concatenate_datasets(datasets_ok)
train_raw = train_raw.shuffle(seed=SEED)

splits = train_raw.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_ds = splits["train"]
eval_ds = splits["test"]

print(f"Train rows: {len(train_ds)}")
print(f"Eval rows: {len(eval_ds)}")
train_ds[0]

In [ ]:
SYSTEM_PROMPT = (
    "You are an SVG code generator. "
    "When given a description, output ONLY a single valid SVG with these rules:\n"
    "1. Fill the full viewBox — shapes should be large and centered, not small or tucked into a corner.\n"
    "2. Use solid fills and simple strokes. No masks, filters, or external references.\n"
    "3. End output with </svg>.\n"
)

def format_sft_text(example):
    if not example["prompt"] or not example["svg"]:
        return {"text": ""}
    svg = example["svg"].strip().replace("\n", "")

    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{svg}<|im_end|>\n"
    )
    return {"text": text}


train_text = train_ds.map(format_sft_text, remove_columns=train_ds.column_names)
train_text = train_text.filter(lambda x: x["text"] != "")

eval_text = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)
eval_text = eval_text.filter(lambda x: x["text"] != "")

In [ ]:
# CHECK FIRST VALUE IN DATASET
print(train_text[0]["text"])
print(len(train_text[0]["text"]))
print(len(tokenizer.encode(train_text[0]["text"])))

In [ ]:
# CHECK RANDOM VALUE IN DATASET
train_text_length = len(train_text)
ran_idx = random.randint(1, train_text_length - 1)

print(train_text[ran_idx]["text"])
print(len(train_text[ran_idx]["text"]))
print(len(tokenizer.encode(train_text[ran_idx]["text"])))

In [ ]:
def analyze_dataset_metrics(dataset, name, tokenizer):
    print(f"--- Metrics for {name} ---")

    char_counts = []
    token_counts = []
    path_counts = []

    for example in dataset:
        full_text = example["text"]
        char_counts.append(len(full_text))
        token_counts.append(len(tokenizer.encode(full_text)))

        # Extract SVG from the full formatted text and count paths
        svg_start_idx = full_text.rfind("<svg")
        svg_end_idx = full_text.rfind("</svg>")

        if svg_start_idx != -1 and svg_end_idx != -1 and svg_end_idx > svg_start_idx:
            svg_content = full_text[svg_start_idx : svg_end_idx + len("</svg>")]
            if (svg_content.count("<path")):
              path_counts.append(svg_content.count("<path"))

    if char_counts:
        print(f"Characters (full formatted text):")
        print(f"  Max: {max(char_counts):.0f}, Avg: {sum(char_counts) / len(char_counts):.2f}")
    if token_counts:
        print(f"Tokens (full formatted text):")
        print(f"  Max: {max(token_counts):.0f}, Avg: {sum(token_counts) / len(token_counts):.2f}")
    if path_counts:
        print(f"Path Counts (extracted SVG):")
        print(f"  Max: {max(path_counts):.0f}, Avg: {sum(path_counts) / len(path_counts):.2f}, Num: {len(path_counts)}")
    print("-" * (len(name) + 12))

analyze_dataset_metrics(train_text, "train_text", tokenizer)
analyze_dataset_metrics(eval_text, "eval_text", tokenizer)

### **Section 5: Training the Model**

> Run the training for the model

> (Running ignore warnings is **optional** if any Future Warnings appear that are cluttering the output cell block)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth.chat_templates import train_on_responses_only

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    average_tokens_across_devices=False,
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_steps=CONFIG["warmup_steps"],
    #warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
    #gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=False,
    args=training_args,
    assistant_only_loss=True, # MASKING
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>system\n",
    response_part = "<|im_start|>assistant\n",
)

train_result = trainer.train()
train_result

### **Section 6: Saving the Model**

> Save the pre-trained model for future use

> You can local save, save to google drive or saved to HuggingFace with a **WRITE** API Key

> (Uncomment respective codeblock for whichever you want to save to. You can also save to all three)


In [ ]:
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"]) # LOCAL SAVING
tokenizer.save_pretrained(CONFIG["output_dir"]) # LOCAL SAVING

print(f"Saved adapter + tokenizer to: {CONFIG['output_dir']}")

In [ ]:
# CLOUD SAVING TO GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')

model.save_pretrained("/content/drive/MyDrive/svg_model_midterm") # SAVE FILE TO DRIVE
tokenizer.save_pretrained("/content/drive/MyDrive/svg_model_midterm") # SAVE FILE TO DRIVE

In [ ]:
# CLOUD SAVING TO HF
api_key = userdata.get("HF_TOKEN_WRITE") # USE HF TOKEN KEY WITH WRITING PERMISSIONS
os.environ["HF_TOKEN_WRITE"] = api_key # USE HF TOKEN KEY WITH WRITING PERMISSIONS

model.push_to_hub("tk2558/qwen_lora_midterm", token = api_key) # Online saving to HF
tokenizer.push_to_hub("tk2558/qwen_lora_midterm", token = api_key) # Online saving to HF

### **Section 7 (OPTIONAL): Unloading Model**

> You can unload your trained model from where you saved it. This is helpful for reusing trained model for the future.

> You can skip to this part and uncomment the codeblock below if you are using pretrained model provided

> (May need to run Section 3 to make sure you have access to Hugging Face API Key)

In [ ]:
# UNLOADING PRE-SAVED MODEL FROM FILE LOCATION

# from unsloth import FastLanguageModel
# from transformers import AutoModel, AutoTokenizer

# from google.colab import drive
# drive.mount('/content/drive')

# MODEL_PATH = "tk2558/qwen_lora_midterm" # Load Model from Hugging Face Example PATH

# tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
# model = AutoModel.from_pretrained(MODEL_PATH, dtype="auto")

### **Section 8: Generating Output Preparation**

> Initialize functions to prepare for model to start generating outputs. Make sure that the model's outputs are VALID and have a fallback SVG just in case

> (Also make sure model is using GPU)

In [ ]:
SVG_REGEX = re.compile(r"<svg.*?</svg>", re.IGNORECASE | re.DOTALL)

def clean_output(text):
    if "assistant" in text:
        text = text.split("assistant")[-1]
    return text.strip()

def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""


def is_valid_svg(svg_text):
    if not svg_text:
        #print("Invalid: No SVG Text")
        return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.endswith("svg")
    except ET.ParseError:
        print("Invalid: Parse Error")
        return False

def extract_svg_features(text):
    color = "black" # DEFAULT
    shape = "circle" # DEFAULT

    color_match = re.search(r'#(?:[0-9a-fA-F]{3}){1,2}', text)
    if color_match:
        color = color_match.group(0)

    if "<rect" in text:
        shape = "rect"
    elif "<ellipse" in text:
        shape = "ellipse"

    return color, shape

def fallback_svg(prompt, failed_output=""):
    color, shape = extract_svg_features(failed_output)
    base = '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
    bg = '<rect x="0" y="0" width="256" height="256" fill="white"/>'

    if shape == "rect":
        obj = f'<rect x="64" y="64" width="128" height="128" fill="{color}"/>'

    elif shape == "ellipse":
        obj = f'<ellipse cx="128" cy="128" rx="80" ry="50" fill="{color}"/>'

    else: # shape == path, circle or something else
        obj = f'<circle cx="128" cy="128" r="64" fill="{color}"/>'

    return base + bg + obj + '</svg>'

In [ ]:
def clean_test_prompt(prompt: str) -> str:
    # 1. Fix encoding garbage (ÃƒÂ‚Ã‚Â etc.)
    try:
        import ftfy
        prompt = ftfy.fix_text(prompt)
    except ImportError: # fallback without ftfy — catches the most common double-encoding
        try:
            prompt = prompt.encode('latin-1').decode('utf-8')
        except (UnicodeDecodeError, UnicodeEncodeError):
            pass

    # 2. Remove meta-instructions that leaked into prompts
    meta_patterns = [
        r"don'?t use markdown[^.]*\.?",
        r"just give svg code[^.]*\.?",
    ]
    for pattern in meta_patterns:
        prompt = re.sub(pattern, '', prompt, flags=re.IGNORECASE)

    # 3. Remove leftover punctuation/whitespace from deletions
    prompt = re.sub(r'\s+', ' ', prompt).strip().strip('.,;:')
    #print(prompt)
    return prompt

In [ ]:
print(torch.cuda.is_available()) # CHECK GPU AVAILABLE
print(model.device) # CHECK GPU IN USE BY MODEL

In [ ]:
model.eval() # SET MODEL TO EVALUATION MODE
FastLanguageModel.for_inference(model) # FAST INFERENCE MODE

### **Section 9: Generating Outputs**

> Time to Generate Outputs!

> If you want to generate output one prompt at a time, use generate_svg. You can uncomment out text_streamer and streamer=text_streamer if you want to see the output generate in real time.

> If you want to generate outputs from a batch of prompts, use generate_batch_svg.

> We set sample to false for Greedy Generation for faster outputs

In [ ]:
from transformers import TextStreamer

SYSTEM_PROMPT = (
    "You are an SVG code generator. "
    "When given a description, output ONLY a single valid SVG with these rules:\n"
    "1. Fill the full viewBox — shapes should be large and centered, not small or tucked into a corner.\n"
    "2. Use solid fills and simple strokes. No masks, filters, or external references.\n"
    "3. End output with </svg>.\n"
)

def generate_svg(prompt, max_new_tokens=1024):
    input_text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{clean_test_prompt(prompt)}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
    text_streamer = TextStreamer(tokenizer, skip_prompt = True)

    #with torch.no_grad():
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False, # GREEDY
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            streamer = text_streamer,
            repetition_penalty=1.1,
        )

    generated_tokens = output_ids[0][inputs.input_ids.shape[1]:]
    decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    decoded = clean_output(decoded)
    svg = extract_svg(decoded)

    if not is_valid_svg(svg):
        #print("Invalid")
        svg = fallback_svg(prompt, decoded)

    return svg

test_time = time.time()
test_prompt = "A stylized red icon depicting a document with a pencil beside it."
pred_svg = generate_svg(test_prompt)
elapsed_time = (time.time() - test_time) # Test Time it takes for One Generation
print(pred_svg[:500])
print("Valid SVG:", is_valid_svg(pred_svg))
print(elapsed_time)
print(f"{len(pred_svg)/elapsed_time:.1f} tok/s")

In [ ]:
def generate_batch_svg(prompts, max_new_tokens=2048):
    input_texts = [
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{clean_test_prompt(prompt)}<|im_end|>\n"
        "<|im_start|>assistant\n"
        for prompt in prompts
    ]

    inputs = tokenizer(
        text=input_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=CONFIG["max_seq_length"]
    ).to(model.device)

    #text_streamer = TextStreamer(tokenizer, skip_prompt = True)

    with torch.inference_mode():  # upgraded from no_grad
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.1,
            #streamer = text_streamer,
        )

    batch_svgs = []
    input_length = inputs.input_ids.shape[1]

    for i, prompt in enumerate(prompts):
        generated_tokens = output_ids[i][input_length:]

        # Strip padding tokens from the end
        non_pad = (generated_tokens != tokenizer.pad_token_id).nonzero()
        if len(non_pad) > 0:
            generated_tokens = generated_tokens[:non_pad[-1].item() + 1]

        decoded = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
        decoded = clean_output(decoded)
        svg = extract_svg(decoded)

        if not is_valid_svg(svg):
            svg = fallback_svg(prompt, decoded)

        batch_svgs.append(svg)

    return batch_svgs

# --- Test ---
# DEBUG_PROMPTS = [
#     "firewood stack cut logs wood with leaf illustration.",
#     "The image shows five horizontal lines of varying thicknesses and lengths, arranged vertically on a white background.",
#     "A stylized icon depicting a curved arrow within a square shape. Don't use markdown just give svg code",
#     "The image contains black geometric shapes against a white background, forming an abstract representation of a person sitting on a chair.",
#     "The image shows a single dark gray triangle pointing upwards, centered against a plain white background.",
# ]



# # Warmup — first generation is always slower due to CUDA kernel init
# print("Warming up...")
# _ = generate_batch_svg(DEBUG_PROMPTS[:1])

# # Test batch sizes
# for batch_size in [5]:
#     prompts = DEBUG_PROMPTS[:batch_size]
#     t0 = time.time()
#     results = generate_batch_svg(prompts)
#     elapsed = time.time() - t0

#     valid = sum(1 for svg in results if is_valid_svg(svg))
#     print(f"Batch {batch_size:2d} | {elapsed:.1f}s total | {elapsed/batch_size:.1f}s per SVG | {valid}/{batch_size}")

### **Section 10: Testing Prompts**

> Test model against the 1000 prompts in test.csv and generate a csv submission for Kaggle

> Two Options:
> *  Use First Codeblock for testing prompts one at a time
> *  Use Second Codeblock for testing prompts in batches

> If submission.csv already exists and accessible, it will pick up from where it left off

> (Make sure clean_training_prompt enable)

In [ ]:
# Submission generation scaffold: expects Kaggle prompt file with columns `id,prompt`.
# SINGLE PROMPT VERSION

# TEST_PROMPTS_PATH = "/kaggle/input/svg-test-public-prompts/test_prompts.csv
#SUBMISSION_PATH = "/kaggle/working/submission.csv"

# TEST_PROMPTS_PATH = "/content/test.csv"
# SUBMISSION_PATH = "/content/submission.csv"

# test_df = pd.read_csv(TEST_PROMPTS_PATH)

# total = len(test_df)
# checkpoint = total // 10  # every 10%
# SAVE_EVERY = 10  # save every N samples

# rows = []
# invalid_count = 0
# t0 = time.time()

# # If resuming, load existing file
# if os.path.exists(SUBMISSION_PATH):
#     existing_df = pd.read_csv(SUBMISSION_PATH)
#     rows = existing_df.to_dict("records")
#     start_idx = len(rows)
#     print(f"Resuming from {start_idx}...")
# else:
#     print(f"Starting from the first...")
#     start_idx = 0

# # Slice off already-processed rows
# remaining_df = test_df.iloc[start_idx:].reset_index(drop=True)
# prompts = remaining_df["prompt"].tolist()
# ids = remaining_df["id"].tolist()

# for _, row in test_df.iterrows():
#     prompt = clean_test_prompt(row["prompt"])
#     svg = generate_svg(prompt)
#     if not is_valid_svg(svg):
#         invalid_count += 1
#         svg = fallback_svg(prompt)

#     rows.append({"id": row["id"], "svg": svg})

#     if (_ + 1) % checkpoint == 0:
#         percent = int(((_ + 1) / total) * 100)
#         print(f"{percent}% done ({_+1}/{total})")

#     if (_ + 1) % SAVE_EVERY == 0:
#         print("Saving")
#         pd.DataFrame(rows).to_csv(SUBMISSION_PATH, index=False)

# sub_df = pd.DataFrame(rows)
# sub_df.to_csv(SUBMISSION_PATH, index=False)

# elapsed_min = (time.time() - t0) / 60
# print(f"Saved: {SUBMISSION_PATH}")
# print(f"Rows: {len(sub_df)}")
# print(f"Invalid/fallback count: {invalid_count}")
# print(f"Invalid SVGs: {invalid_count}/{total}")
# print(f"Runtime (minutes): {elapsed_min:.2f}")
# sub_df.head()

In [ ]:
# Submission generation scaffold: expects Kaggle prompt file with columns `id,prompt`.
# BATCH VERSION

TEST_PROMPTS_PATH = "/content/test.csv"
SUBMISSION_PATH = "/content/submission.csv"

test_df = pd.read_csv(TEST_PROMPTS_PATH)

total = len(test_df)
BATCH_SIZE = 5
SAVE_EVERY = 10  # in batches

rows = []
t0 = time.time()

if os.path.exists(SUBMISSION_PATH):
    existing_df = pd.read_csv(SUBMISSION_PATH)
    rows = existing_df.to_dict("records")
    start_idx = len(rows)
    print(f"Resuming from {start_idx}...")
else:
    print("Starting from the first...")
    start_idx = 0

# Slice off already-processed rows
remaining_df = test_df.iloc[start_idx:].reset_index(drop=True)

prompts = remaining_df["prompt"].tolist()
ids     = remaining_df["id"].tolist()

for batch_num, batch_start in enumerate(range(0, len(prompts), BATCH_SIZE)):
    batch_prompts = prompts[batch_start : batch_start + BATCH_SIZE]
    batch_ids     = ids[batch_start : batch_start + BATCH_SIZE]

    batch_svgs = generate_batch_svg(batch_prompts)
    print("Batch Completed")
    for id_, svg in zip(batch_ids, batch_svgs):
        rows.append({"id": id_, "svg": svg})

    if (batch_num + 1) % SAVE_EVERY == 0:
        pd.DataFrame(rows).to_csv(SUBMISSION_PATH, index=False)
        done = start_idx + batch_start + len(batch_prompts)
        print(f"{done}/{total} done")

# Final save
sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)

elapsed_min = (time.time() - t0) / 60
print(f"Saved: {SUBMISSION_PATH}")
print(f"Rows: {len(sub_df)}")
print(f"Runtime (minutes): {elapsed_min:.2f}")
sub_df.head()

### **Section 11: Links**

> [Midterm Report in ACL Format]

> [Model Weights in Hugging Face](https://huggingface.co/tk2558/qwen_lora_midterm)

> [Github Repo](https://github.com/tk2558/Deep-Learning-Text-to-SVG-Generation)